# 🪑 CHAIR Benchmark Evaluation with ONLY (ICCV'25)
### Environment: Kaggle 2× NVIDIA T4 GPUs | Full Precision (BF16 / FP16 Tensor Cores)

This notebook evaluates **CHAIR** (Caption Hallucination Assessment with Image Relevance) on Multimodal LLMs using the **ONLY** intervention method:
- **LLaVA-1.5-7B** (`llava-hf/llava-1.5-7b-hf`)
- **Qwen2-VL-7B-Instruct** (`Qwen/Qwen2-VL-7B-Instruct`)

---
### 📌 Benchmark Protocol & Requirements:
1. **Latency Benchmark (10 samples)**: Run with `max_samples = 10` using `import time` (`time.perf_counter()` + `torch.cuda.synchronize()`) to measure precise GPU latency, calculate average time per sample, and estimate full benchmark duration.
2. **Full Evaluation (500 samples)**: Evaluates 500 COCO val2014 images sampled with **seed 2027** (`selected_chair_val2014_seed2027.json`).
3. **Prompt & Decoding**: Prompt `"Describe this image."` with greedy decoding (`do_sample=False`), strictly `max_new_tokens=128`.
4. **Official Standalone Metric**: Evaluation using **Maxlinn/CHAIR-metric-standalone** ([GitHub Repository](https://github.com/Maxlinn/CHAIR-metric-standalone/tree/main)) computing `CHAIRs`, `CHAIRi`, `Recall`, and `Caption Length` via pre-cached `chair.pkl`.

In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Anti-Conflict Dependency Installation (MANDATORY)
# ==============================================================================
# 1. Gỡ bỏ torchaudio để tránh 100% xung đột CUDA mismatch trên Kaggle
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết cho HuggingFace transformers, Qwen2-VL, và CHAIR metric
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    nltk \
    huggingface_hub \
    pandas

# 3. Tải các gói từ vựng NLTK cần thiết cho CHAIR evaluator (Maxlinn standalone)
import nltk
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

# 4. Xác nhận dependency hợp lệ
import torch, transformers, accelerate, qwen_vl_utils, nltk
print(f"✅ Dependencies Verified! PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()}) | Transformers: {transformers.__version__}")

In [ ]:
# ==============================================================================
# CELL 2: HuggingFace Authentication (Optional - via Kaggle Secret 'HF_TOKEN')
# ==============================================================================
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace Hub login successful with Kaggle Secret 'HF_TOKEN'!")
except Exception as e:
    print(f"ℹ️ Note: Could not authenticate via Kaggle Secrets ({e}).")
    print("   Public models (llava-hf/llava-1.5-7b-hf, Qwen/Qwen2-VL-7B-Instruct) load without token.")

In [ ]:
# ==============================================================================
# CELL 3: Clone Repository ONLY & Set Working Directory
# ==============================================================================
import os
import subprocess

REPO_URL = "https://github.com/ntmy12/ONLY.git"
REPO_DIR = "/kaggle/working/ONLY"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Clone completed successfully!")
else:
    print(f"Repository already exists at {REPO_DIR}.")

%cd {REPO_DIR}
!pwd

In [ ]:
# ==============================================================================
# CELL 4: Hardware & GPU Environment Verification
# ==============================================================================
import torch

num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  GPU {i}: {name} ({mem_gb:.1f} GB VRAM)")

bf16_ok = torch.cuda.is_bf16_supported()
print(f"BF16 Hardware Support: {'✅ YES (Native)' if bf16_ok else '⚡ FP16 Tensor Cores (Optimized for T4)'}")

In [ ]:
# ==============================================================================
# CELL 5: Verify COCO val2014 Images, Manifest (Seed 2027), & CHAIR Cache
# ==============================================================================
import sys, os, json
sys.path.insert(0, "/kaggle/working/ONLY")
sys.path.insert(0, "/kaggle/working/ONLY/eval_bench")

from eval_bench.eval_chair import auto_detect_coco_dir, resolve_chair_cache, resolve_chair_samples

# 1. Auto-detect COCO val2014 directory
try:
    coco_dir = auto_detect_coco_dir()
    print(f"✅ COCO val2014 images found: {coco_dir}")
except Exception as e:
    print(f"⚠️ {e}")

# 2. Check CHAIR manifest (500 samples, seed 2027)
samples_500 = resolve_chair_samples(coco_dir, num_samples=500, seed=2027)
print(f"✅ CHAIR manifest loaded: {len(samples_500)} images (seed 2027)")
print(f"   Sample image [0]: {samples_500[0]}")

# 3. Check CHAIR evaluator cache (Maxlinn chair.pkl)
chair_cache = resolve_chair_cache()
print(f"✅ Maxlinn CHAIR standalone cache: {chair_cache}")

In [ ]:
# ==============================================================================
# CELL 6: Latency Benchmark on 10 Samples (Precise Timing via import time)
# ==============================================================================
# Model choice: 'llava' (LLaVA-1.5-7B) or 'qwen2vl' (Qwen2-VL-7B-Instruct)
MODEL_CHOICE = "llava"
USE_ONLY = True

print(f"⏱️ Running Latency Benchmark on 10 samples for {MODEL_CHOICE.upper()} (ONLY={USE_ONLY})...")
print("   Prompt: 'Describe this image.' | max_new_tokens = 128 | time.perf_counter()\n")

!python eval_bench/eval_chair.py \
    --model {MODEL_CHOICE} \
    --use_only {USE_ONLY} \
    --max_samples 10 \
    --prompt "Describe this image." \
    --max_new_tokens 128 \
    --seed 2027 \
    --precision auto \
    --device_map auto \
    --out_dir ./results/{MODEL_CHOICE}_chair_latency_10samples

In [ ]:
# ==============================================================================
# CELL 7: Full CHAIR Benchmark Evaluation on 500 Images (Seed 2027)
# ==============================================================================
# Select model and method
MODEL_CHOICE = "llava"  # change to 'qwen2vl' for Qwen2-VL-7B-Instruct
USE_ONLY = True         # set to False for Baseline without ONLY

mode_name = "only" if USE_ONLY else "baseline"
print(f"🎯 Running Full CHAIR Benchmark (500 images) for {MODEL_CHOICE.upper()} - {mode_name.upper()}...")
print("   Prompt: 'Describe this image.' | max_new_tokens = 128 | seed = 2027\n")

!python eval_bench/eval_chair.py \
    --model {MODEL_CHOICE} \
    --use_only {USE_ONLY} \
    --max_samples 500 \
    --prompt "Describe this image." \
    --max_new_tokens 128 \
    --seed 2027 \
    --precision auto \
    --device_map auto \
    --out_dir ./results/{MODEL_CHOICE}_chair_{mode_name}_500samples

In [ ]:
# ==============================================================================
# CELL 8: Summary Metrics Display & CHAIR Results Table
# ==============================================================================
import glob, os, json
import pandas as pd

# Find all summary metrics files in ./results
summary_files = sorted(glob.glob("./results/*/summary_metrics.json"), key=os.path.getmtime)

if summary_files:
    rows = []
    for sf in summary_files:
        with open(sf, "r", encoding="utf-8") as f:
            d = json.load(f)
        m = d.get("metrics", {})
        t = d.get("timing", {})
        
        chairs = m.get("CHAIRs", 0.0)
        chairs = chairs * 100 if chairs <= 1.0 else chairs
        chairi = m.get("CHAIRi", 0.0)
        chairi = chairi * 100 if chairi <= 1.0 else chairi
        recall = m.get("Recall", 0.0)
        recall = recall * 100 if recall <= 1.0 else recall
        cap_len = m.get("Len", m.get("Caption_Length", 0.0))
        
        rows.append({
            "Model": d.get("model", "").upper(),
            "Method": d.get("mode", "").upper(),
            "Samples": t.get("total_samples", m.get("samples_tested", 0)),
            "CHAIRs (%)": round(chairs, 2),
            "CHAIRi (%)": round(chairi, 2),
            "Recall (%)": round(recall, 2),
            "Avg Cap Len": round(cap_len, 2),
            "Avg Time/Sample (s)": round(t.get("avg_time_per_sample_s", 0.0), 4),
            "Total Time (s)": round(t.get("total_inference_time_s", 0.0), 2),
            "Throughput (it/s)": round(t.get("throughput_samples_per_sec", 0.0), 2),
        })
    
    df = pd.DataFrame(rows)
    print("\n" + "=" * 95)
    print("                   CHAIR BENCHMARK - CONSOLIDATED RESULTS")
    print("=" * 95)
    display(df)
else:
    print("No summary_metrics.json found in ./results yet.")

In [ ]:
# ==============================================================================
# CELL 9: Qualitative Inspection of Generated Captions & Hallucinations
# ==============================================================================
import glob, os, json

details_files = sorted(glob.glob("./results/*/chair_details.json"), key=os.path.getmtime)
if details_files:
    latest_details = details_files[-1]
    print(f"Inspecting qualitative details from: {latest_details}\n")
    with open(latest_details, "r", encoding="utf-8") as f:
        data = json.load(f)
    sentences = data.get("sentences", [])[:5]  # first 5 samples
    for idx, s in enumerate(sentences):
        print(f"[Sample {idx+1}] Image ID: {s.get('image_id')}")
        print(f"  Caption: {s.get('caption')}")
        print(f"  GT Objects    : {s.get('mscoco_gt_words')}")
        print(f"  Gen Objects   : {s.get('mscoco_generated_words')}")
        print(f"  Hallucinated  : {s.get('mscoco_hallucinated_words')}")
        print(f"  CHAIRs: {s.get('metrics', {}).get('CHAIRs')} | CHAIRi: {s.get('metrics', {}).get('CHAIRi'):.2f}")
        print("-" * 80)
else:
    print("No chair_details.json found yet.")